**Data Source**: https://www.kaggle.com/competitions/titanic/data

In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = SparkConf()
conf.set('spark.master', 'local[6]')
conf.set('spark.app.name', 'Spark ML Scratch')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [2]:
train_raw = spark.read.csv('data/titanic/train.csv', header = True, inferSchema = True)

train_raw.show(5, truncate = False)
train_raw.printSchema()
train_raw.count()

+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|Name                                               |Sex   |Age |SibSp|Parch|Ticket          |Fare   |Cabin|Embarked|
+-----------+--------+------+---------------------------------------------------+------+----+-----+-----+----------------+-------+-----+--------+
|1          |0       |3     |Braund, Mr. Owen Harris                            |male  |22.0|1    |0    |A/5 21171       |7.25   |NULL |S       |
|2          |1       |1     |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|38.0|1    |0    |PC 17599        |71.2833|C85  |C       |
|3          |1       |3     |Heikkinen, Miss. Laina                             |female|26.0|0    |0    |STON/O2. 3101282|7.925  |NULL |S       |
|4          |1       |1     |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|35.0|1    |0    |113803          |53

891

In [3]:
import pyspark.sql.functions as f

train = train_raw.withColumn('name_temp', f.split(f.col('Name'), ',')[1])\
            .withColumn('Status', f.trim(f.split(f.col('name_temp'), '\.')[0]))\
            .drop('PassengerId', 'Name', 'Ticket', 'Cabin', 'name_temp')
train.show(5)

+--------+------+------+----+-----+-----+-------+--------+------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|Status|
+--------+------+------+----+-----+-----+-------+--------+------+
|       0|     3|  male|22.0|    1|    0|   7.25|       S|    Mr|
|       1|     1|female|38.0|    1|    0|71.2833|       C|   Mrs|
|       1|     3|female|26.0|    0|    0|  7.925|       S|  Miss|
|       1|     1|female|35.0|    1|    0|   53.1|       S|   Mrs|
|       0|     3|  male|35.0|    0|    0|   8.05|       S|    Mr|
+--------+------+------+----+-----+-----+-------+--------+------+
only showing top 5 rows


In [4]:
train.select(['*']).summary().show()

+-------+-------------------+------------------+------+------------------+------------------+-------------------+-----------------+--------+------------+
|summary|           Survived|            Pclass|   Sex|               Age|             SibSp|              Parch|             Fare|Embarked|      Status|
+-------+-------------------+------------------+------+------------------+------------------+-------------------+-----------------+--------+------------+
|  count|                891|               891|   891|               714|               891|                891|              891|     889|         891|
|   mean| 0.3838383838383838| 2.308641975308642|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|    NULL|        NULL|
| stddev|0.48659245426485753|0.8360712409770491|  NULL|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|    NULL|        NULL|
|    min|                  0|                 1|female|              0.42|  

## Transformer

### Vector Assembler

In [5]:
from pyspark.ml.feature import VectorAssembler

inputCols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch']
assembler = VectorAssembler(inputCols = inputCols, outputCol = 'vector_feature', handleInvalid = 'skip')
assembler.transform(train).show(5, truncate = False)

+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
|Survived|Pclass|Sex   |Age |SibSp|Parch|Fare   |Embarked|Status|vector_feature        |
+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
|0       |3     |male  |22.0|1    |0    |7.25   |S       |Mr    |[0.0,3.0,22.0,1.0,0.0]|
|1       |1     |female|38.0|1    |0    |71.2833|C       |Mrs   |[1.0,1.0,38.0,1.0,0.0]|
|1       |3     |female|26.0|0    |0    |7.925  |S       |Miss  |[1.0,3.0,26.0,0.0,0.0]|
|1       |1     |female|35.0|1    |0    |53.1   |S       |Mrs   |[1.0,1.0,35.0,1.0,0.0]|
|0       |3     |male  |35.0|0    |0    |8.05   |S       |Mr    |[0.0,3.0,35.0,0.0,0.0]|
+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
only showing top 5 rows


### Imputer

In [7]:
from pyspark.ml.feature import Imputer

num_imputer = Imputer(inputCol = 'Age', outputCol = 'Age_Imp', strategy = 'median')
cat_imputer = Imputer(inputCol = 'Embarked', outputCol = 'Embarked_Imp', strategy = 'mode')

In [8]:
num_model = num_imputer.fit(train)
num_model.transform(train).filter('Age IS NULL').show(5)

+--------+------+------+----+-----+-----+------+--------+------+-------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|  Fare|Embarked|Status|Age_Imp|
+--------+------+------+----+-----+-----+------+--------+------+-------+
|       0|     3|  male|NULL|    0|    0|8.4583|       Q|    Mr|   28.0|
|       1|     2|  male|NULL|    0|    0|  13.0|       S|    Mr|   28.0|
|       1|     3|female|NULL|    0|    0| 7.225|       C|   Mrs|   28.0|
|       0|     3|  male|NULL|    0|    0| 7.225|       C|    Mr|   28.0|
|       1|     3|female|NULL|    0|    0|7.8792|       Q|  Miss|   28.0|
+--------+------+------+----+-----+-----+------+--------+------+-------+
only showing top 5 rows


In [9]:
cat_model = cat_imputer.fit(train)
cat_model.transform(train).filter('Embarked IS NULL').show(5)

IllegalArgumentException: requirement failed: Column Embarked must be of type numeric but was actually of type string.

In [10]:
train.select('Embarked').groupBy('Embarked').count().show()

+--------+-----+
|Embarked|count|
+--------+-----+
|       Q|   77|
|    NULL|    2|
|       C|  168|
|       S|  644|
+--------+-----+



### Scaler

In [11]:
from pyspark.ml.feature import StandardScaler, MinMaxScaler

num_inputCols = ['Age', 'Fare']
num_outputCols = ['Age_Sclaed', 'Fare_Scaled']

num_assembler = VectorAssembler(inputCols = num_inputCols, outputCol = 'num_vector', handleInvalid = 'skip')

standard = StandardScaler(inputCol = 'num_vector', outputCol = 'scaled')
minmax = MinMaxScaler(inputCol = 'num_vector', outputCol = 'scaled')

vectorized = num_assembler.transform(train)

standard_model = standard.fit(vectorized)
minmax_model = minmax.fit(vectorized)

In [12]:
standard_model.transform(vectorized.select('num_vector')).show(5, truncate = False)

+--------------+----------------------------------------+
|num_vector    |scaled                                  |
+--------------+----------------------------------------+
|[22.0,7.25]   |[1.5144738264626911,0.13700201550092822]|
|[38.0,71.2833]|[2.6159093366173756,1.3470283822837676] |
|[26.0,7.925]  |[1.7898327040013622,0.14975737556480773]|
|[35.0,53.1]   |[2.4093901784633722,1.0034216583585225] |
|[35.0,8.05]   |[2.4093901784633722,0.152119479280341]  |
+--------------+----------------------------------------+
only showing top 5 rows


In [13]:
minmax_model.transform(vectorized.select('num_vector')).show(5, truncate = False)

+--------------+------------------------------------------+
|num_vector    |scaled                                    |
+--------------+------------------------------------------+
|[22.0,7.25]   |[0.2711736617240513,0.014151057562208049] |
|[38.0,71.2833]|[0.4722292033174164,0.13913573538264068]  |
|[26.0,7.925]  |[0.32143754712239253,0.015468569817999833]|
|[35.0,53.1]   |[0.43453128926866047,0.10364429745562033] |
|[35.0,8.05]   |[0.43453128926866047,0.015712553569072387]|
+--------------+------------------------------------------+
only showing top 5 rows


### Encoder

In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

inputCols = ['Sex', 'Status']
outputCols = ['Sex_Idx', 'Status_Idx']

indexer = StringIndexer(inputCols = inputCols, outputCols = outputCols, handleInvalid = 'keep')
indexer_model = indexer.fit(train)

In [15]:
indexed = indexer_model.transform(train.select('Sex', 'Status'))
indexed.show(5)

+------+------+-------+----------+
|   Sex|Status|Sex_Idx|Status_Idx|
+------+------+-------+----------+
|  male|    Mr|    0.0|       0.0|
|female|   Mrs|    1.0|       2.0|
|female|  Miss|    1.0|       1.0|
|female|   Mrs|    1.0|       2.0|
|  male|    Mr|    0.0|       0.0|
+------+------+-------+----------+
only showing top 5 rows


In [16]:
onehot = OneHotEncoder(inputCols = inputCols, outputCols = outputCols, handleInvalid = 'keep')

onehot_model = onehot.fit(train)

IllegalArgumentException: requirement failed: Column Sex must be of type numeric but was actually of type string.

In [17]:
onehot = OneHotEncoder(inputCols = outputCols, outputCols = ['Sex_OneHot', 'Status_OneHot'], handleInvalid = 'keep')

onehot_model = onehot.fit(indexed)
onehot_model.transform(indexed).show(5)

+------+------+-------+----------+-------------+--------------+
|   Sex|Status|Sex_Idx|Status_Idx|   Sex_OneHot| Status_OneHot|
+------+------+-------+----------+-------------+--------------+
|  male|    Mr|    0.0|       0.0|(3,[0],[1.0])|(18,[0],[1.0])|
|female|   Mrs|    1.0|       2.0|(3,[1],[1.0])|(18,[2],[1.0])|
|female|  Miss|    1.0|       1.0|(3,[1],[1.0])|(18,[1],[1.0])|
|female|   Mrs|    1.0|       2.0|(3,[1],[1.0])|(18,[2],[1.0])|
|  male|    Mr|    0.0|       0.0|(3,[0],[1.0])|(18,[0],[1.0])|
+------+------+-------+----------+-------------+--------------+
only showing top 5 rows


## Estimator

In [18]:
test_raw = spark.read.csv('data/titanic/test.csv', header = True, inferSchema = True)
test_label = spark.read.csv('data/titanic/gender_submission.csv', header = True, inferSchema = True)

test_with_label = test_raw.join(test_label, 'PassengerId', how = 'inner')
test_with_label.show(5)

+-----------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+--------+
|PassengerId|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|Survived|
+-----------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+--------+
|        892|     3|    Kelly, Mr. James|  male|34.5|    0|    0| 330911| 7.8292| NULL|       Q|       0|
|        893|     3|Wilkes, Mrs. Jame...|female|47.0|    1|    0| 363272|    7.0| NULL|       S|       1|
|        894|     2|Myles, Mr. Thomas...|  male|62.0|    0|    0| 240276| 9.6875| NULL|       Q|       0|
|        895|     3|    Wirz, Mr. Albert|  male|27.0|    0|    0| 315154| 8.6625| NULL|       S|       0|
|        896|     3|Hirvonen, Mrs. Al...|female|22.0|    1|    1|3101298|12.2875| NULL|       S|       1|
+-----------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+--------+
only showing top 5 rows


In [19]:
test = test_with_label.withColumn('name_temp', f.split(f.col('Name'), ',')[1])\
            .withColumn('Status', f.trim(f.split(f.col('name_temp'), '\.')[0]))\
            .drop('PassengerId', 'Name', 'Ticket', 'Cabin', 'name_temp')

test.select(['*']).summary().show()

+-------+------------------+------+------------------+------------------+------------------+------------------+--------+-------------------+------+
|summary|            Pclass|   Sex|               Age|             SibSp|             Parch|              Fare|Embarked|           Survived|Status|
+-------+------------------+------+------------------+------------------+------------------+------------------+--------+-------------------+------+
|  count|               418|   418|               332|               418|               418|               417|     418|                418|   418|
|   mean|2.2655502392344498|  NULL|30.272590361445783|0.4473684210526316|0.3923444976076555|  35.6271884892086|    NULL|0.36363636363636365|  NULL|
| stddev|0.8418375519640503|  NULL|14.181209235624424|0.8967595611217135|0.9814288785371694|55.907576179973844|    NULL| 0.4816221409322309|  NULL|
|    min|                 1|female|              0.17|                 0|                 0|               0.0| 

In [20]:
imputer = Imputer(inputCols = ['Age', 'Fare'], outputCols = ['Age_Imp', 'Fare_Imp'], strategy = 'median')
imputer_model = imputer.fit(train)

assembler = VectorAssembler(inputCols = ['Pclass', 'Age_Imp', 'SibSp', 'Parch', 'Fare_Imp'], outputCol = 'features')

In [21]:
from pyspark.ml.classification import RandomForestClassifier

rf_clf = RandomForestClassifier(featuresCol = 'features',
                                labelCol = 'Survived',
                                predictionCol = 'prediction',
                                seed = 42,
                                maxDepth = 5,
                                maxBins = 32,
                                numTrees = 20)

train_data = imputer_model.transform(train)
train_data = assembler.transform(train_data)

rf_model = rf_clf.fit(train_data)

test_data = imputer_model.transform(test)
test_data = assembler.transform(test_data)

prediction = rf_model.transform(test_data)

In [57]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(predictionCol = 'prediction', labelCol = 'Survived', metricName = 'accuracy')
evaluator.evaluate(prediction)

0.6698564593301436

## Pipeline

In [23]:
train = train_raw.na.fill({'Embarked': 'C'})
train = train.withColumn('name_temp', f.split(f.col('Name'), ',')[1])\
            .withColumn('Status', f.trim(f.split(f.col('name_temp'), '\.')[0]))\
            .drop('PassengerId', 'Name', 'Ticket', 'Cabin', 'name_temp')
test = test_raw.join(test_label, 'PassengerId', how = 'inner')
test = test.na.fill({'Embarked': 'C'})
test = test.withColumn('name_temp', f.split(f.col('Name'), ',')[1])\
            .withColumn('Status', f.trim(f.split(f.col('name_temp'), '\.')[0]))\
            .drop('PassengerId', 'Name', 'Ticket', 'Cabin', 'name_temp')

cat_inputCols = ['Sex', 'Embarked', 'Status']
cat_outputCols = ['Sex_Idx', 'Embarked_Idx', 'Status_Idx']
indexer = StringIndexer(inputCols = cat_inputCols, outputCols = cat_outputCols, handleInvalid = 'keep')

num_inputCols = ['Age', 'Fare']
num_outputCols = ['Age_Imp', 'Fare_Imp']
num_imputer = Imputer(inputCols = num_inputCols, outputCols = num_outputCols, strategy = 'median')
num_assembler = VectorAssembler(inputCols = num_outputCols, outputCol = 'num_vector')
num_scaler = StandardScaler(inputCol = 'num_vector', outputCol = 'scaled_features')

fin_inputCols = ['Pclass', 'SibSp', 'Parch'] + cat_outputCols + ['scaled_features']
fin_assembler = VectorAssembler(inputCols = fin_inputCols, outputCol = 'features')

rf_clf = RandomForestClassifier(
    featuresCol = 'features',
    labelCol = 'Survived',
    predictionCol = 'prediction',
    seed = 42
)

In [24]:
from pyspark.ml.pipeline import Pipeline

pipeline = Pipeline(stages = [
    indexer,
    num_imputer,
    num_assembler,
    num_scaler,
    fin_assembler,
    rf_clf
])

model = pipeline.fit(train)
fin_pred = model.transform(test)

evaluator.evaluate(fin_pred)

0.9401913875598086

## HyperParameter Tuning

In [61]:
from pyspark.ml.tuning import ParamGridBuilder

paramGrid = (ParamGridBuilder()
             .addGrid(rf_clf.maxDepth, [5, 6, 7])
             .addGrid(rf_clf.maxBins, [32, 48, 64])
             .addGrid(rf_clf.numTrees, [20, 25, 30])
             .build())

In [62]:
from pyspark.ml.tuning import TrainValidationSplit

tvs = TrainValidationSplit(
    estimator = pipeline,
    estimatorParamMaps = paramGrid,
    evaluator = evaluator,
    trainRatio = 0.8
)

# tvs_model = tvs.fit(train)

In [63]:
from pyspark.ml.tuning import CrossValidator

cv = CrossValidator(
    estimator = pipeline,
    estimatorParamMaps = paramGrid,
    evaluator = evaluator,
    numFolds = 5,
)

cv_model = cv.fit(train)

In [64]:
cv_pred = cv_model.transform(test)

evaluator.evaluate(cv_pred)

0.9282296650717703

In [65]:
best_rf_model = cv_model.bestModel.stages[-1]

print(best_rf_model.getOrDefault('maxDepth'))
print(best_rf_model.getOrDefault('maxBins'))
print(best_rf_model.getOrDefault('numTrees'))

6
32
30
